In [ ]:
!pip install genesis-world tensordict

from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

## 歩行モデルを使った、ロボットの位置変化予測モデル

入力(前後速度、左右速度、角速度)に対し、ロボットがどれほど移動したか(dx, dy, dr=角度変化)を予測するモデル。
z軸の移動は想定しない

dim3 - 線形モデル - dim3

In [ ]:

WORKSPACE = Path("/content/drive/MyDrive/PhysicalAI_adv/loc")
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

TRAIN_NAME = "thefirstloc"
DATA_DIR = WORKSPACE / TRAIN_NAME

TRAIN_DATA = DATA_DIR / "train_data.jsonl"
EVAL_DATA = DATA_DIR / "eval_data.jsonl"
MODEL_DATA = DATA_DIR / "model.json"

NUM_ENVS = 32
NUM_STEPS = 1e5

WALKER_MODEL = Path("/content/drive/MyDrive/PhysicalAI_adv/dog/thefirststep/weights/actor.pth")

## 角度は全部radianです

"""
train_data.jsonlの中身

{id:int,command:(vx,vy,vr), d_pos:(dx,dy,dr), init_pos:(x,y,r), end_pos:(x,y,r)}
idは連番にすること!

eval_data.jsonlの中身

(train_data.jsonlと同じ)

model.jsonの中身

weight: np.array(3,3) 
bias: np.array(3,)
configs: {
    data_num:int
}
"""

: 

In [ ]:
import numpy as np

class SimpleLinear:
    def __init__(self) -> None:
        self.W = np.ones((3,3))
        self.b = np.zeros((3,))

    def predict(self, X):
        return X @ self.W + self.b

    def fit(self, X, y):
        X = np.atleast_2d(np.asarray(X, dtype=np.float64))
        y = np.asarray(y, dtype=np.float64)
        A = np.concatenate((X, np.ones((X.shape[0], 1))), axis=1)
        theta, *_ = np.linalg.lstsq(A, y, rcond=None)
        self.W = theta[:-1]
        self.b = theta[-1]

: 

In [ ]:
import torch
from torch import nn

class SimpleWalker(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.mean = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, output_dim)
        )
        self.log_std = nn.Parameter(torch.full((output_dim,), 0.01 ))

    def forward(self, x):
        mean = self.mean(x)
        return torch.distributions.Normal(mean, torch.exp(self.log_std))



In [ ]:
import math
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "models" / "walker.py").exists():
    ROOT = ROOT.parent
if (ROOT / "models" / "walker.py").exists() and str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import genesis as gs
from genesis.utils.geom import inv_quat, quat_to_xyz, transform_by_quat, transform_quat_by_quat


class Go2Env:
    """コマンドを固定して並列に歩かせ、区間の開始姿勢と終了姿勢を返す。

    速度コマンドは ``set_commands`` で与える (lin_vel_x, lin_vel_y, ang_vel_yaw)。
    ``step`` は1制御ステップで、転倒してもリセットしない。
    ``reset`` では環境ごとに x, y を ±2 m、ヨーを ±π の一様分布でずらす。
    """

    xy_range = 2.0
    yaw_range = torch.pi

    def __init__(self, num_envs, policy, env_cfg=None, obs_cfg=None, show_viewer=False, rendered_envs_idx=None):
        if env_cfg is None:
            env_cfg = ENV_CFG
        if obs_cfg is None:
            obs_cfg = OBS_CFG
        self.policy = policy.to(gs.device)
        self.policy.eval()
        self.num_envs: int = num_envs
        self.num_actions = env_cfg["num_actions"]
        self.num_commands = 3
        self.cfg = env_cfg
        self.device = gs.device

        self.simulate_action_latency = True  # there is a 1 step latency on real robot
        self.dt = 0.02  # control frequency on real robot is 50hz
        self.max_episode_length = math.ceil(env_cfg["episode_length_s"] / self.dt)

        self.env_cfg = env_cfg
        self.obs_cfg = obs_cfg
        self.obs_scales: dict[str, float] = obs_cfg["obs_scales"]

        if show_viewer and rendered_envs_idx is None:
            rendered_envs_idx = (0,)
        if rendered_envs_idx is not None:
            rendered_envs_idx = tuple(rendered_envs_idx)
            if len(rendered_envs_idx) == 0 or any(idx < 0 or idx >= num_envs for idx in rendered_envs_idx):
                raise ValueError(f"rendered_envs_idx must be in [0, {num_envs}), got {rendered_envs_idx}")
        # follow_entity tracks env 0's simulated pose, which matches the drawn pose only when
        # that env is unshifted. A grid of envs is centered at the origin instead.
        follow_env0 = show_viewer and rendered_envs_idx == (0,)
        base_pos = env_cfg["base_init_pos"]
        if follow_env0:
            # Used as an offset from the robot once the camera starts following.
            camera_pos = (1.5, -1.5, 0.9)
            camera_lookat = (base_pos[0], base_pos[1], base_pos[2])
        else:
            n_shown = len(rendered_envs_idx) if rendered_envs_idx is not None else max(num_envs, 1)
            cols = math.ceil(math.sqrt(n_shown))
            rows = math.ceil(n_shown / cols)
            span = max(rows - 1, cols - 1, 1) * 2.0
            dist = max(3.0, span)
            camera_pos = (dist, -dist, max(1.5, dist * 0.45))
            camera_lookat = (0.0, 0.0, base_pos[2])

        self.scene = gs.Scene(
            sim_options=gs.options.SimOptions(
                dt=self.dt,
                substeps=2,
            ),
            rigid_options=gs.options.RigidOptions(
                enable_self_collision=False,
                # For this locomotion policy, there are usually no more than 20 collision pairs. Setting a low value
                # can save memory. Violating this condition will raise an exception.
                max_collision_pairs=20,
                tolerance=1e-5,
            ),
            viewer_options=gs.options.ViewerOptions(
                camera_pos=camera_pos,
                camera_lookat=camera_lookat,
                camera_fov=40,
            ),
            vis_options=gs.options.VisOptions(
                show_world_frame=True,
                rendered_envs_idx=rendered_envs_idx,
            )
            if show_viewer
            else None,
            show_viewer=show_viewer,
        )

        self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/plane/plane.urdf",
                fixed=True,
            )
        )

        self.robot = self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/go2/urdf/go2.urdf",
                pos=self.env_cfg["base_init_pos"],
                quat=self.env_cfg["base_init_quat"],
            ),
        )

        self.scene.build(
            n_envs=num_envs,
            env_spacing=(0.0, 0.0) if follow_env0 or not show_viewer else (2.0, 2.0),
            center_envs_at_origin=not follow_env0,
        )
        if follow_env0:
            self.scene.viewer.follow_entity(self.robot)

        # names to indices
        self.motors_dof_idx = torch.tensor(
            [self.robot.get_joint(name).dof_start for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_int,
            device=gs.device,
        )
        self.actions_dof_idx = torch.argsort(self.motors_dof_idx)

        # PD control parameters
        self.robot.set_dofs_kp([self.env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
        self.robot.set_dofs_kv([self.env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)

        # Define global gravity direction vector
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], dtype=gs.tc_float, device=gs.device)

        # Initial state
        self.init_base_pos = torch.tensor(self.env_cfg["base_init_pos"], dtype=gs.tc_float, device=gs.device)
        self.init_base_quat = torch.tensor(self.env_cfg["base_init_quat"], dtype=gs.tc_float, device=gs.device)
        self.inv_base_init_quat = inv_quat(self.init_base_quat)
        self.init_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][joint.name] for joint in self.robot.joints[1:]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.init_qpos = torch.concatenate((self.init_base_pos, self.init_base_quat, self.init_dof_pos))
        self.init_projected_gravity = transform_by_quat(self.global_gravity, self.inv_base_init_quat)

        self.base_lin_vel = torch.zeros((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_ang_vel = torch.zeros((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.projected_gravity = self.init_projected_gravity.repeat(self.num_envs, 1)
        self.episode_length_buf = torch.zeros((self.num_envs,), dtype=gs.tc_int, device=gs.device)
        self.commands = torch.zeros((self.num_envs, self.num_commands), dtype=gs.tc_float, device=gs.device)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            device=gs.device,
            dtype=gs.tc_float,
        )
        self.actions = torch.zeros((self.num_envs, self.num_actions), dtype=gs.tc_float, device=gs.device)
        self.last_actions = torch.zeros_like(self.actions)
        self.dof_pos = self.init_dof_pos.repeat(self.num_envs, 1)
        self.dof_vel = torch.zeros_like(self.actions)
        self.base_pos = self.init_base_pos.repeat(self.num_envs, 1)
        self.base_quat = self.init_base_quat.repeat(self.num_envs, 1)
        self.base_euler = torch.zeros((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.default_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][name] for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.obs_buf = torch.empty((self.num_envs, 0), dtype=gs.tc_float, device=gs.device)

        self.reset()

    def set_commands(self, commands):
        """Set velocity commands. Shape ``(3,)`` is broadcast to every env, or pass ``(num_envs, 3)``."""
        commands = torch.as_tensor(commands, dtype=gs.tc_float, device=self.device)
        if commands.ndim == 1:
            if commands.shape != (self.num_commands,):
                raise ValueError(f"expected command shape {(self.num_commands,)}, got {tuple(commands.shape)}")
            commands = commands.unsqueeze(0).expand(self.num_envs, -1)
        elif commands.shape != (self.num_envs, self.num_commands):
            raise ValueError(
                f"expected command shape {(self.num_envs, self.num_commands)}, got {tuple(commands.shape)}"
            )
        self.commands.copy_(commands)
        self._update_observation()

    def step(self, actions):
        actions = torch.as_tensor(actions, dtype=gs.tc_float, device=self.device)
        self.actions.copy_(actions.clamp(-self.env_cfg["clip_actions"], self.env_cfg["clip_actions"]))
        exec_actions = self.last_actions if self.simulate_action_latency else self.actions
        target_dof_pos = exec_actions * self.env_cfg["action_scale"] + self.default_dof_pos
        self.robot.control_dofs_position(target_dof_pos[:, self.actions_dof_idx], slice(6, 18))
        # Refreshes the viewer. With it open, steps are paced to scene.viewer.realtime_factor (1.0).
        self.scene.step()

        self.episode_length_buf += 1
        self.base_pos = self.robot.get_pos()
        self.base_quat = self.robot.get_quat()
        self.base_euler = quat_to_xyz(
            transform_quat_by_quat(self.inv_base_init_quat, self.base_quat), rpy=True, degrees=True
        )
        inv_base_quat = inv_quat(self.base_quat)
        self.base_lin_vel = transform_by_quat(self.robot.get_vel(), inv_base_quat)
        self.base_ang_vel = transform_by_quat(self.robot.get_ang(), inv_base_quat)
        self.projected_gravity = transform_by_quat(self.global_gravity, inv_base_quat)
        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)
        self.dof_vel = self.robot.get_dofs_velocity(self.motors_dof_idx)

        done = self.episode_length_buf > self.max_episode_length
        done |= torch.abs(self.base_euler[:, 1]) > self.env_cfg["termination_if_pitch_greater_than"]
        done |= torch.abs(self.base_euler[:, 0]) > self.env_cfg["termination_if_roll_greater_than"]
        done |= self.scene.rigid_solver.get_error_envs_mask()

        self._update_observation()
        self.last_actions.copy_(self.actions)

        return self.obs_buf.clone(), done

    def get_observations(self):
        return self.obs_buf.clone()

    def _reset_idx(self, envs_idx=None):
        self.robot.set_qpos(self.init_qpos, envs_idx=envs_idx, zero_velocity=True, skip_forward=True)

        if envs_idx is None:
            self.base_pos.copy_(self.init_base_pos)
            self.base_quat.copy_(self.init_base_quat)
            self.projected_gravity.copy_(self.init_projected_gravity)
            self.dof_pos.copy_(self.init_dof_pos)
            self.base_lin_vel.zero_()
            self.base_ang_vel.zero_()
            self.dof_vel.zero_()
            self.actions.zero_()
            self.last_actions.zero_()
            self.episode_length_buf.zero_()
        else:
            torch.where(envs_idx[:, None], self.init_base_pos, self.base_pos, out=self.base_pos)
            torch.where(envs_idx[:, None], self.init_base_quat, self.base_quat, out=self.base_quat)
            torch.where(
                envs_idx[:, None], self.init_projected_gravity, self.projected_gravity, out=self.projected_gravity
            )
            torch.where(envs_idx[:, None], self.init_dof_pos, self.dof_pos, out=self.dof_pos)
            self.base_lin_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.base_ang_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_actions.masked_fill_(envs_idx[:, None], 0.0)
            self.episode_length_buf.masked_fill_(envs_idx, 0)

    def _update_observation(self):
        self.obs_buf = torch.cat(
            [
                self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
                self.projected_gravity,  # 3
                self.commands * self.commands_scale,  # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
                self.dof_vel * self.obs_scales["dof_vel"],  # 12
                self.actions,  # 12
            ],
            dim=-1,
        )

    def reset(self, envs_idx=None, xy_range=None, yaw_range=None):
        """立ち姿勢に戻し、環境ごとに平面位置とヨーをずらす。z と関節角は共通。"""
        self._reset_idx(envs_idx)
        self._update_observation()
        obs = self.obs_buf.clone()
        xy_range = self.xy_range if xy_range is None else xy_range
        yaw_range = self.yaw_range if yaw_range is None else yaw_range
        if xy_range == 0 and yaw_range == 0:
            return obs
        if envs_idx is None:
            n = self.num_envs
        else:
            n = int(envs_idx.sum().item())
            if n == 0:
                return obs
        pos = self.init_base_pos.expand(n, -1).clone()
        pos[:, 0] += (torch.rand(n, device=self.device) * 2 - 1) * xy_range
        pos[:, 1] += (torch.rand(n, device=self.device) * 2 - 1) * xy_range
        yaw = (torch.rand(n, device=self.device) * 2 - 1) * yaw_range
        half = yaw * 0.5
        quat = torch.stack((half.cos(), torch.zeros_like(half), torch.zeros_like(half), half.sin()), dim=-1)
        qpos = torch.cat((pos, quat, self.init_dof_pos.expand(n, -1)), dim=-1)
        self.robot.set_qpos(qpos, envs_idx=envs_idx, zero_velocity=True, skip_forward=True)
        if envs_idx is None:
            self.base_pos.copy_(pos)
            self.base_quat.copy_(quat)
        else:
            self.base_pos[envs_idx] = pos
            self.base_quat[envs_idx] = quat
        self._update_observation()
        return self.obs_buf.clone()

    def _xy_yaw(self):
        euler = quat_to_xyz(
            transform_quat_by_quat(self.inv_base_init_quat, self.base_quat),
            rpy=True,
            degrees=False,
        )
        pos = self.base_pos[:, :2].detach().cpu().numpy()
        yaw = euler[:, 2].detach().cpu().numpy()
        return pos, yaw

    @torch.no_grad()
    def rollout(self, commands, duration_s):
        """``commands`` は (3,) または (num_envs, 3): 前後速度、左右速度、ヨー角速度 [rad/s]。

        返り値の位置は (num_envs, 2) の x, y [m]。ヨーは (num_envs,) [rad]。
        """
        n_steps = int(round(duration_s / self.dt))
        if n_steps < 1:
            raise ValueError(f"duration_s must cover at least one step ({self.dt} s)")
        self.set_commands(commands)
        start_pos, start_yaw = self._xy_yaw()
        obs = self.get_observations()
        for _ in range(n_steps):
            action = self.policy(obs).mean
            obs, _done = self.step(action)
        end_pos, end_yaw = self._xy_yaw()
        return {
            "start_pos": start_pos,
            "start_yaw": start_yaw,
            "end_pos": end_pos,
            "end_yaw": end_yaw,
        }

# 学習ノートの Go2Env と同じ関節・観測スケール。角度はラジアン。
ENV_CFG = {
    "num_actions": 12,
    "default_joint_angles": {
        "FL_hip_joint": 0.0,
        "FR_hip_joint": 0.0,
        "RL_hip_joint": 0.0,
        "RR_hip_joint": 0.0,
        "FL_thigh_joint": 0.8,
        "FR_thigh_joint": 0.8,
        "RL_thigh_joint": 1.0,
        "RR_thigh_joint": 1.0,
        "FL_calf_joint": -1.5,
        "FR_calf_joint": -1.5,
        "RL_calf_joint": -1.5,
        "RR_calf_joint": -1.5,
    },
    "joint_names": [
        "FR_hip_joint",
        "FR_thigh_joint",
        "FR_calf_joint",
        "FL_hip_joint",
        "FL_thigh_joint",
        "FL_calf_joint",
        "RR_hip_joint",
        "RR_thigh_joint",
        "RR_calf_joint",
        "RL_hip_joint",
        "RL_thigh_joint",
        "RL_calf_joint",
    ],
    "kp": 20.0,
    "kd": 0.5,
    "termination_if_roll_greater_than": 30,
    "termination_if_pitch_greater_than": 30,
    "base_init_pos": [0.0, 0.0, 0.42],
    "base_init_quat": [1.0, 0.0, 0.0, 0.0],
    "episode_length_s": 20.0,
    "resampling_time_s": 4.0,
    "action_scale": 0.25,
    "simulate_action_latency": True,
    "clip_actions": 100.0,
}
OBS_CFG = {
    "obs_scales": {
        "lin_vel": 2.0,
        "ang_vel": 0.25,
        "dof_pos": 1.0,
        "dof_vel": 0.05,
    },
}


def load_policy(path=None):
    path = Path(path) if path is not None else ROOT / "weights" / "thefirststep" / "policy.pth"
    policy = SimpleWalker(45, 12)
    policy.load_state_dict(torch.load(path, weights_only=True, map_location="cpu"))
    policy.eval()
    return policy


def make_env(num_envs, policy_path=None, show_viewer=False):
    try:
        gs.init(backend=gs.gpu)
    except Exception as exc:
        if "already initialized" not in str(exc).lower():
            raise
    return Go2Env(num_envs, load_policy(policy_path), show_viewer=show_viewer)


In [ ]:
## Data Collection
import json

import numpy as np
import torch

vx_range = [-0.5, 1.0]
vy_range = [-0.5, 0.5]
vr_range = [-1.5, 1.5]

def command_gen(lower, upper, batch_shape, zero_prob):
    assert lower.shape == upper.shape
    shape = (*batch_shape, *lower.shape)
    out = (upper - lower) * torch.rand(shape, dtype=gs.tc_float, device=gs.device) + lower
    zero = torch.rand((*batch_shape, 1), device=gs.device) < zero_prob
    rands =  out.masked_fill(zero, 0.0)

    return rands

try:
    gs.init(backend=gs.gpu)
except Exception as exc:
    if "already initialized" not in str(exc).lower():
        raise

walker_model = SimpleWalker(45, 12)
walker_state = torch.load(WALKER_MODEL, weights_only=True, map_location="cpu")
walker_model.load_state_dict(walker_state)
env = Go2Env(NUM_ENVS, walker_model)

# 歩行学習でコマンドを保ち続ける時間。d_pos はこの区間の変位を開始ヨー基準の機体座標で表す。
DURATION_S = env.cfg["resampling_time_s"]
ZERO_PROB = 0.1
EVAL_RATIO = 0.1

n_total = int(NUM_STEPS)
n_eval = int(round(n_total * EVAL_RATIO))
n_train = n_total - n_eval
lower = torch.tensor([vx_range[0], vy_range[0], vr_range[0]], dtype=gs.tc_float, device=gs.device)
upper = torch.tensor([vx_range[1], vy_range[1], vr_range[1]], dtype=gs.tc_float, device=gs.device)
roll_limit = env.cfg["termination_if_roll_greater_than"]
pitch_limit = env.cfg["termination_if_pitch_greater_than"]

DATA_DIR.mkdir(parents=True, exist_ok=True)
counts = {"train": 0, "eval": 0}
batches = 0
max_batches = (n_total // NUM_ENVS + 1) * 20

with TRAIN_DATA.open("w") as train_f, EVAL_DATA.open("w") as eval_f:
    files = {"train": train_f, "eval": eval_f}
    while counts["train"] < n_train or counts["eval"] < n_eval:
        batches += 1
        if batches > max_batches:
            raise RuntimeError(
                f"転倒を除いても {n_total} 件に届かない (train {counts['train']}, eval {counts['eval']})"
            )
        env.reset()
        commands = command_gen(lower, upper, (NUM_ENVS,), ZERO_PROB)
        result = env.rollout(commands, DURATION_S)

        upright = (env.base_euler[:, 0].abs() <= roll_limit) & (env.base_euler[:, 1].abs() <= pitch_limit)
        upright &= ~env.scene.rigid_solver.get_error_envs_mask()
        keep = upright.detach().cpu().numpy()

        delta = result["end_pos"] - result["start_pos"]
        yaw = result["start_yaw"]
        c = np.cos(yaw)
        s = np.sin(yaw)
        d_pos = np.stack(
            (
                c * delta[:, 0] + s * delta[:, 1],
                -s * delta[:, 0] + c * delta[:, 1],
                (result["end_yaw"] - yaw + np.pi) % (2 * np.pi) - np.pi,
            ),
            axis=-1,
        )
        command_np = commands.detach().cpu().numpy()

        for i in range(NUM_ENVS):
            if not keep[i]:
                continue
            train_left = n_train - counts["train"]
            eval_left = n_eval - counts["eval"]
            if train_left <= 0 and eval_left <= 0:
                break
            if eval_left <= 0 or (train_left > 0 and np.random.rand() >= eval_left / (train_left + eval_left)):
                split = "train"
            else:
                split = "eval"
            record = {
                "id": counts[split],
                "command": command_np[i].tolist(),
                "d_pos": d_pos[i].tolist(),
                "init_pos": [float(result["start_pos"][i, 0]), float(result["start_pos"][i, 1]), float(yaw[i])],
                "end_pos": [
                    float(result["end_pos"][i, 0]),
                    float(result["end_pos"][i, 1]),
                    float(result["end_yaw"][i]),
                ],
            }
            files[split].write(json.dumps(record) + "\n")
            counts[split] += 1
        if batches % 50 == 0:
            print(f"train {counts['train']}/{n_train}  eval {counts['eval']}/{n_eval}", flush=True)

print(f"wrote {counts['train']} -> {TRAIN_DATA}")
print(f"wrote {counts['eval']} -> {EVAL_DATA}")


In [ ]:
## Train
import json

def load_xy(path):
    command, d_pos = [], []
    with path.open() as f:
        for line in f:
            row = json.loads(line)
            command.append(row["command"])
            d_pos.append(row["d_pos"])
    return np.asarray(command, dtype=np.float64), np.asarray(d_pos, dtype=np.float64)

X_train, y_train = load_xy(TRAIN_DATA)
model = SimpleLinear()
model.fit(X_train, y_train)

MODEL_DATA.parent.mkdir(parents=True, exist_ok=True)
MODEL_DATA.write_text(json.dumps({
    "weight": model.W.tolist(),
    "bias": model.b.tolist(),
    "configs": {"data_num": int(X_train.shape[0])},
}))
print(f"n={X_train.shape[0]}")
print("W")
print(model.W)
print("b", model.b)
print(f"saved {MODEL_DATA}")


In [ ]:
## Eval
import json

with MODEL_DATA.open() as f:
    saved = json.load(f)
model = SimpleLinear()
model.W = np.asarray(saved["weight"], dtype=np.float64)
model.b = np.asarray(saved["bias"], dtype=np.float64)

X_eval, y_eval = load_xy(EVAL_DATA)
pred = model.predict(X_eval)
err = pred - y_eval
mae = np.mean(np.abs(err), axis=0)
rmse = np.sqrt(np.mean(err ** 2, axis=0))

print(f"n={X_eval.shape[0]}  train_data_num={saved['configs']['data_num']}")
for name, axis_mae, axis_rmse in zip(("dx", "dy", "dr"), mae, rmse):
    print(f"{name}: mae={axis_mae:.4f}  rmse={axis_rmse:.4f}")
print(f"all: mae={mae.mean():.4f}  rmse={np.sqrt(np.mean(err ** 2)):.4f}")
